# 04 - Global Results Comparison for Sparse-View CT

Compare TpV and ResUNet on the full processed test set.

## 1. Environment Setup and Imports

Set paths, imports, device, and output locations.

In [ ]:
!pip install astra-toolbox

from google.colab import drive

drive.mount("/content/drive")

import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed2"
TPV_PARAMS_PATH = PROJECT_ROOT / "outputs" / "tpv" / "tpv_params.json"
WEIGHTS_DIR = PROJECT_ROOT / "weights" / "unet"
RESUNET_BEST_CHECKPOINT_PATH = WEIGHTS_DIR / "resunet_generalized_best.pt"
RESUNET_LATEST_CHECKPOINT_PATH = WEIGHTS_DIR / "resunet_generalized_latest.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import models, operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities.metrics import PSNR, SSIM

SEED = 42
MAX_TEST_SLICES_PER_PATIENT = None  # Set to 1 for a quick smoke test.

np.random.seed(SEED)
torch.manual_seed(SEED)
device = utilities.get_device()

print("Device used:", device)
print("Processed data directory:", PROCESSED_DIR)
print("TpV parameters path:", TPV_PARAMS_PATH)
print("Comparison outputs directory:", OUTPUT_DIR)

## 2. Load Data Contract and TpV Parameters

Load the processed manifest, test patient list, and TpV parameters exported by notebook 01.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

if not TPV_PARAMS_PATH.exists():
    raise FileNotFoundError(
        f"Missing TpV parameter file: {TPV_PARAMS_PATH}. Run notebook 01_TpV_reconstruction.ipynb first."
    )

with TPV_PARAMS_PATH.open("r", encoding="utf-8") as file:
    tpv_params = json.load(file)

required_tpv_params = {"lmbda", "p", "maxiter", "tolf", "tolx"}
missing_tpv_params = required_tpv_params.difference(tpv_params)
if missing_tpv_params:
    raise KeyError(f"Missing TpV parameters in {TPV_PARAMS_PATH}: {sorted(missing_tpv_params)}")

lmbda = float(tpv_params["lmbda"])
p = float(tpv_params["p"])
maxiter = int(tpv_params["maxiter"])
tolf = float(tpv_params["tolf"])
tolx = float(tpv_params["tolx"])

if not (0.1 < p < 0.5):
    raise ValueError(f"Project specifications require 0.1 < p < 0.5. Got p = {p}")

IMAGE_SIZE = int(manifest["image_size"])
DETECTOR_SIZE = int(manifest["detector_size"])
GEOMETRY = manifest["geometry"]
ANGLE_COUNTS = tuple(int(n_views) for n_views in manifest["angle_counts"])
angle_keys = [str(n_views) for n_views in ANGLE_COUNTS]

test_records = manifest["splits"]["test"]["patients"]

print("Loaded TpV parameters:", tpv_params)
print("Test patients:", len(test_records))
print("Angle counts:", ANGLE_COUNTS)
print("Image size:", IMAGE_SIZE)
print("Detector size:", DETECTOR_SIZE)

## 3. Configure Operators and ResUNet

Create CT/FBP/TpV operators and load the best available ResUNet checkpoint.

In [ ]:
projectors = {}
fbp_solvers = {}
tpv_solvers = {}

for angle_key in angle_keys:
    n_views = int(angle_key)
    projector = operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, n_views),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
    )
    projectors[angle_key] = projector
    fbp_solvers[angle_key] = solvers.FBP(projector)
    tpv_solvers[angle_key] = solvers.ChambollePockTpVUnconstrained(projector)

resunet_checkpoint_path = RESUNET_BEST_CHECKPOINT_PATH if RESUNET_BEST_CHECKPOINT_PATH.exists() else RESUNET_LATEST_CHECKPOINT_PATH
if not resunet_checkpoint_path.exists():
    raise FileNotFoundError(
        f"No ResUNet checkpoint found. Expected {RESUNET_BEST_CHECKPOINT_PATH} or {RESUNET_LATEST_CHECKPOINT_PATH}."
    )

checkpoint = torch.load(resunet_checkpoint_path, map_location=device)
checkpoint_config = checkpoint.get("config", {})
if "MODEL_FINAL_ACTIVATION" not in checkpoint_config:
    raise RuntimeError(
        "ResUNet checkpoint is missing MODEL_FINAL_ACTIVATION in config. Re-run notebook 02_ResUnet_reconstruction.ipynb."
    )

base_channels = int(checkpoint_config.get("BASE_CHANNELS", 32))
model_final_activation = checkpoint_config["MODEL_FINAL_ACTIVATION"]

resunet = models.UNet(
    ch_in=1,
    ch_out=1,
    middle_ch=(base_channels, 2 * base_channels, 4 * base_channels, 8 * base_channels, 16 * base_channels),
    n_layers_per_block=2,
    down_layers=("ResDownBlock", "ResDownBlock", "ResDownBlock", "ResDownBlock"),
    up_layers=("ResUpBlock", "ResUpBlock", "ResUpBlock", "ResUpBlock"),
    n_heads=None,
    final_activation=model_final_activation,
).to(device)
resunet.load_state_dict(checkpoint["model_state_dict"])
resunet.eval()

print(f"Loaded ResUNet checkpoint from epoch {checkpoint.get('epoch', 'unknown')}: {resunet_checkpoint_path}")

## 4. Full Test Evaluation

Run TpV and ResUNet for every test slice and every sparse-view setting.

In [ ]:
METHOD_TPV = "TpV"
METHOD_RESUNET = "ResUNet"

metrics_rows = []
representative_examples = {}


def compute_metrics(prediction: torch.Tensor, target: torch.Tensor) -> dict[str, float]:
    prediction = torch.clamp(prediction.detach(), 0.0, 1.0)
    target = torch.clamp(target.detach().to(prediction.device), 0.0, 1.0)
    return {
        "psnr": float(PSNR(prediction, target)),
        "ssim": float(SSIM(prediction, target)),
    }


def append_metric_row(patient_id: str, source_path: str, slice_idx: int, angle_key: str, method: str, metrics: dict[str, float]) -> None:
    metrics_rows.append(
        {
            "patient_id": patient_id,
            "source_path": source_path,
            "slice_idx": int(slice_idx),
            "views": int(angle_key),
            "method": method,
            "psnr": metrics["psnr"],
            "ssim": metrics["ssim"],
        }
    )


total_test_slices = sum(
    min(record["num_samples"], int(MAX_TEST_SLICES_PER_PATIENT))
    if MAX_TEST_SLICES_PER_PATIENT is not None
    else record["num_samples"]
    for record in test_records
)
with tqdm(total=total_test_slices, desc="TpV", unit="slice", position=0) as tpv_bar, tqdm(
    total=total_test_slices, desc="ResUNet", unit="slice", position=1
) as resunet_bar:
    for patient_record in test_records:
        patient_path = PROCESSED_DIR / patient_record["path"]
        payload = torch.load(patient_path, map_location="cpu")

        clean_images = payload["clean"].to(torch.float32)
        sinograms = {key: val.to(torch.float32) for key, val in payload["sinograms"].items()}
        source_paths = payload.get("source_paths", [])
        metadata = payload["metadata"]
        patient_id = metadata["patient_id"]

        slice_indices = list(range(clean_images.shape[0]))
        if MAX_TEST_SLICES_PER_PATIENT is not None:
            slice_indices = slice_indices[: int(MAX_TEST_SLICES_PER_PATIENT)]

        print(f"Evaluating patient {patient_id}: {len(slice_indices)} slices")

        for slice_idx in slice_indices:
            x_true_cpu = clean_images[slice_idx : slice_idx + 1]
            x_true = x_true_cpu.to(device)
            source_path = source_paths[slice_idx] if slice_idx < len(source_paths) else ""

            for angle_key in angle_keys:
                y_delta_cpu = sinograms[angle_key][slice_idx : slice_idx + 1]
                y_delta = y_delta_cpu.to(device)

                x_tpv, _ = tpv_solvers[angle_key](
                    y_delta_cpu,
                    lmbda=lmbda,
                    starting_point=None,
                    x_true=x_true_cpu,
                    maxiter=maxiter,
                    tolf=tolf,
                    tolx=tolx,
                    p=p,
                    verbose=False,
                )
                x_tpv_norm = normalize(x_tpv.detach()).clamp(0.0, 1.0).to(torch.float32)
                tpv_metrics = compute_metrics(x_tpv_norm, x_true_cpu)
                append_metric_row(patient_id, source_path, slice_idx, angle_key, METHOD_TPV, tpv_metrics)

                x_fbp, _ = fbp_solvers[angle_key](y_delta, x_true=None, starting_point=None)
                x_fbp_norm = normalize(x_fbp.detach()).clamp(0.0, 1.0).to(torch.float32)
                with torch.no_grad():
                    x_resunet = torch.clamp(resunet(x_fbp_norm.to(device)), 0.0, 1.0)
                x_resunet_cpu = x_resunet.detach().cpu().to(torch.float32)
                resunet_metrics = compute_metrics(x_resunet, x_true)
                append_metric_row(patient_id, source_path, slice_idx, angle_key, METHOD_RESUNET, resunet_metrics)

                if angle_key not in representative_examples:
                    representative_examples[angle_key] = {
                        "patient_id": patient_id,
                        "slice_idx": int(slice_idx),
                        "ground_truth": x_true_cpu.clone(),
                        "sinogram": y_delta_cpu.detach().cpu().clone(),
                        METHOD_TPV: x_tpv_norm.detach().cpu().clone(),
                        METHOD_RESUNET: x_resunet_cpu.clone(),
                    }

            tpv_bar.update(1)
            resunet_bar.update(1)

print(f"Computed {len(metrics_rows)} metric rows")


## 5. Save Metrics and Summary

Save per-image metrics and aggregate PSNR/SSIM by method and view count.

In [ ]:
per_image_metrics_path = OUTPUT_DIR / "test_metrics_per_image.csv"
summary_csv_path = OUTPUT_DIR / "test_metrics_summary.csv"
summary_json_path = OUTPUT_DIR / "test_metrics_summary.json"


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def summarize_group(rows: list[dict], scope: str, method: str, views) -> dict:
    psnr_values = np.array([row["psnr"] for row in rows], dtype=float)
    ssim_values = np.array([row["ssim"] for row in rows], dtype=float)
    return {
        "scope": scope,
        "views": views,
        "method": method,
        "num_images": int(len(rows)),
        "mean_psnr": float(psnr_values.mean()),
        "std_psnr": float(psnr_values.std()),
        "mean_ssim": float(ssim_values.mean()),
        "std_ssim": float(ssim_values.std()),
    }

summary_rows = []
for method in [METHOD_TPV, METHOD_RESUNET]:
    method_rows = [row for row in metrics_rows if row["method"] == method]
    summary_rows.append(summarize_group(method_rows, "overall", method, "all"))

    for angle_key in angle_keys:
        view_rows = [row for row in method_rows if row["views"] == int(angle_key)]
        summary_rows.append(summarize_group(view_rows, "per_view", method, int(angle_key)))

write_csv(
    per_image_metrics_path,
    metrics_rows,
    ["patient_id", "source_path", "slice_idx", "views", "method", "psnr", "ssim"],
)
write_csv(
    summary_csv_path,
    summary_rows,
    ["scope", "views", "method", "num_images", "mean_psnr", "std_psnr", "mean_ssim", "std_ssim"],
)

summary_payload = {
    "tpv_params": tpv_params,
    "resunet_checkpoint": str(resunet_checkpoint_path),
    "max_test_slices_per_patient": MAX_TEST_SLICES_PER_PATIENT,
    "summary": summary_rows,
}
with summary_json_path.open("w", encoding="utf-8") as file:
    json.dump(summary_payload, file, indent=2)

print("Saved per-image metrics:", per_image_metrics_path)
print("Saved summary CSV:", summary_csv_path)
print("Saved summary JSON:", summary_json_path)
print("=" * 96)
print(f"{'SCOPE':<10} | {'VIEWS':<8} | {'METHOD':<8} | {'N':<8} | {'PSNR mean':<10} | {'PSNR std':<9} | {'SSIM mean':<10} | {'SSIM std':<9}")
print("=" * 96)
for row in summary_rows:
    print(
        f"{row['scope']:<10} | {str(row['views']):<8} | {row['method']:<8} | {row['num_images']:<8} | "
        f"{row['mean_psnr']:<10.4f} | {row['std_psnr']:<9.4f} | {row['mean_ssim']:<10.4f} | {row['std_ssim']:<9.4f}"
    )
print("=" * 96)

## 6. Representative Visual Panels

Save one representative panel per method and view count.

In [ ]:
for angle_key, example in representative_examples.items():
    gt_img = example["ground_truth"][0, 0].numpy()
    sinogram_img = example["sinogram"][0, 0].squeeze().numpy()

    for method in [METHOD_TPV, METHOD_RESUNET]:
        reconstruction_img = example[method][0, 0].numpy()
        fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
        fig.suptitle(
            f"{method} representative reconstruction - {angle_key} views - "
            f"patient {example['patient_id']} slice {example['slice_idx']}",
            fontsize=13,
        )

        axes[0].imshow(gt_img, cmap="gray", vmin=0.0, vmax=1.0)
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(sinogram_img.T, cmap="gray")
        axes[1].set_title("Noisy Sinogram")
        axes[1].axis("off")

        axes[2].imshow(reconstruction_img, cmap="gray", vmin=0.0, vmax=1.0)
        axes[2].set_title(f"{method} Reconstruction")
        axes[2].axis("off")

        panel_path = OUTPUT_DIR / f"representative_{method.lower()}_{angle_key}_views.png"
        fig.savefig(panel_path, dpi=150)
        plt.show()
        plt.close(fig)
        print("Saved representative panel:", panel_path)

## 7. Aggregate Quantitative Plots

Plot mean PSNR and SSIM across view counts.

In [ ]:
per_view_summary = [row for row in summary_rows if row["scope"] == "per_view"]
view_labels = [str(n_views) for n_views in ANGLE_COUNTS]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
fig.suptitle("Full Test Set Comparison", fontsize=13)

for method, color in [(METHOD_TPV, "tab:blue"), (METHOD_RESUNET, "tab:green")]:
    method_rows = [row for row in per_view_summary if row["method"] == method]
    method_rows = sorted(method_rows, key=lambda row: ANGLE_COUNTS.index(int(row["views"])))
    psnr_values = [row["mean_psnr"] for row in method_rows]
    ssim_values = [row["mean_ssim"] for row in method_rows]

    axes[0].plot(view_labels, psnr_values, marker="o", color=color, label=method)
    axes[1].plot(view_labels, ssim_values, marker="o", color=color, label=method)

axes[0].set_xlabel("Number of views")
axes[0].set_ylabel("Mean PSNR (dB)")
axes[0].set_title("PSNR")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Number of views")
axes[1].set_ylabel("Mean SSIM")
axes[1].set_title("SSIM")
axes[1].set_ylim(0.0, 1.0)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plot_path = OUTPUT_DIR / "test_metrics_summary_plots.png"
fig.savefig(plot_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved aggregate comparison plot:", plot_path)